In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

## QUERTION 2

### Q2. Indexing and searching
### Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

### How does the agentic loop keep calling the model until it stops?

In [5]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [6]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [7]:
question = "How does the agentic loop keep calling the model until it stops?"
answer = llm(question)
print(answer)

The Agentic Loop, also known as the iterative process or loop, is a key component of the decision-making process of some AI systems, including models based on Large Language Models (LLMs) like LLaMA or others in the transformer family. 

In a typical Agentic Loop, there are a few key steps involved:

1. **Initiate a Query**: The model is given a request or prompt, often referred to as a "query" or "prompt" by the user or system administrator.

2. **Generate a Response Candidate**: The model generates a response to this query based on its training data and knowledge.

3. **Evaluate the Response**: The model evaluates its generated response based on some criteria, such as coherence, relevance, or the likelihood that it would be well-received by the user. This stage may also include user feedback through input or other means.

4. **Determine whether the Response is Satisfactory**: If the generated response is deemed acceptable by the model and/or the system administrator or user, then it 

In [8]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [9]:
results = index.search("How does the agentic loop keep calling the model until it stops?")
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [11]:
#question = "I just discovered the course. Can I join now?"



In [13]:
def search(question, course="llm-zoomcamp"):
    return index.search(
        question,
     #   filter_dict={"course": "llm-zoomcamp"},
        num_results=1
    )


In [14]:
search_results = search(question)
print(search_results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG
### Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


### Two solutions are possible:

#### Implement the RAG flow yourself
#### Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
#### Build a RAG over the index from Q2 and answer the query:

### How does the agentic loop keep calling the model until it stops?

In [15]:
from rag_helper import RAGBase
import importlib
import rag_helper
importlib.reload(rag_helper)
#from rag_helper import RAGBase


<module 'rag_helper' from '/app/01-homework/rag_helper.py'>

In [16]:
assistant = RAGBase( index,openai_client)

In [17]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

"I'm ready to answer your questions based on the provided context. Go ahead and ask your question."

In [18]:
answer = assistant.rag('How does the agentic loop keep calling the model until it stops?')

assistant.last_usage

assistant.last_usage.prompt_tokens
assistant.last_usage.completion_tokens
assistant.last_usage.total_tokens

7246